# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR<sup>2</sup> dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. All entities such as record sets and fields are referenced by their `@id`, following the Croissant schema convention.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show metadata description
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below, we print the available record sets (by `@id`), and for each record set, its fields and field `@id`s.

In [ ]:
# List all record sets and their fields from the Croissant schema metadata
from pprint import pprint

print("Available Record Sets and Fields (by @id):\n")

record_set_ids = []

for rset in dataset.metadata.record_sets:
    print(f"- RecordSet @id: {rset['@id']}")
    record_set_ids.append(rset["@id"])
    # List all fields for this record set
    print("  Fields:")
    for field in rset["fields"]:
        print(f"    - {field['name']} (@id: {field['@id']})")
    print()
# The record_set_ids list will be used later for extraction.

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as established above.

Below, we load all available record sets into pandas DataFrames, keyed by their `@id`. We show available columns (field `@id`s) for one selected record set and display its data head.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_set_ids:
    try:
        # records() yields dicts with keys as field @id
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Select the main data record set (first in list or change as needed)
main_record_set_id = record_set_ids[0]
print(f"\nColumns (@id) in main RecordSet {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())

# Show the first few records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
This section applies filtering, normalization, and grouping on the main record set, referencing all fields by their `@id`.
- **Filtering:** We select rows where a protein abundance (use the most relevant numeric field by `@id`) exceeds a threshold.
- **Normalization:** We add a new column, normalized using z-score normalization.
- **Grouping:** We group by a categorical attribute (for example, "protein description" or another experimental group) if present.

Replace `<numeric_field_id>` and `<group_field_id>` with actual `@id`s as needed from the overview above.

In [ ]:
# EXAMPLE: Replace these with the actual field @id from section 2.
# For this FAIR2 dataset, possible numeric fields are abundance, 'cr:abundance', etc.
# To see available fields:
print("Available columns in main record set:")
print(dataframes[main_record_set_id].columns.tolist())

# Let's pick 'cr:abundance' as a numeric field if it exists, else adjust
possible_numeric_fields = [c for c in dataframes[main_record_set_id].columns if 'abundance' in c or 'MW' in c or 'cr:' in c]

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # e.g., 'cr:abundance' or similar
else:
    numeric_field = dataframes[main_record_set_id].select_dtypes('number').columns[0]

# Pick a group field (for example, protein description, accession id, or experimental group)
possible_group_fields = [c for c in dataframes[main_record_set_id].columns if 'desc' in c.lower() or 'group' in c.lower() or 'accession' in c.lower()]
group_field = possible_group_fields[0] if possible_group_fields else None

print(f"\nUsing numeric field @id: {numeric_field}")
if group_field:
    print(f"Using group field @id: {group_field}")

# Filter by a threshold
threshold = dataframes[main_record_set_id][numeric_field].mean()
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows\n")

# Normalize field
norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Sample normalized {numeric_field}:")
print(filtered_df[[numeric_field, norm_col]].head())

# Group by group_field if available
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
    print(f"\nTop grouped means by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Below we visualize the distribution of the selected numeric field (e.g., protein abundance) and the results of the grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Grouped means bar plot (if available)
if group_field is not None and 'grouped_df' in locals():
    plt.figure(figsize=(10, 6))
    sns.barplot(x=group_field, y=numeric_field, data=grouped_df.head(10))
    plt.title(f"Mean {numeric_field} by {group_field} (Top 10)")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" FAIR<sup>2</sup> dataset using the `mlcroissant` library. Each entity (record set, field, column) was referenced by its Croissant `@id` for transparency and reproducibility.

We overviewed schema structure, loaded data records, filtered and normalized a numeric measurement, performed a group aggregation, and visualized data distributions. You can now further analyze, visualize, or prepare this dataset for downstream ML tasks.

For more details or contributions: [FAIR<sup>2</sup> dataset landing page](https://sen.science/doi/10.71728/senscience.vq4a-28xa)